# Qualitative figures — the teaser (Figure 1) and a supplement grid

This notebook renders **real** detector outputs on **real** COCO val images. Nothing is drawn by hand or generated.

For each of about 300 COCO val images, and each model:
- **alone:** the normal batch-1 output;
- **in a batch:** the same image batched with the widest and the tallest image of the pool, so the padding is large (the usual padded batching of the library);
- **in a batch + fix:** our `self` fix for DETR and Faster R-CNN, or the box rescale for YOLOS.

Images are ranked by how *visibly* the detections change (a large box lost or gained, and a clean scene with 2–10 confident objects). The best examples are rendered:
- `teaser.pdf`: Figure 1 (DETR: alone | in a batch | in a batch + fix), full CVPR width;
- `supp_grid.pdf`: 4 models × 2 examples, alone vs. in a batch;
- `contact_<model>.png`: the top candidates for each model, with image ids, so you can pick different examples by hand.

**Picking by hand.** Look at the contact sheets. Choose images without close-up faces of identifiable people, and where the change is obvious at a glance. Put their ids in `TEASER_ID` / `GRID_IDS` in Section 1, then re-run from Section 7 only (the model runs are cached).

**Kaggle:** GPU T4 x2 or P100, Internet ON. About 20–30 minutes.

In [ ]:
!pip install -q "transformers==5.17.0" pycocotools timm

In [ ]:
import os
os.environ.setdefault("PYTORCH_ALLOC_CONF", "expandable_segments:True")   # less memory fragmentation
import io, sys, glob, json, time, math, random, zipfile, platform, datetime, contextlib, urllib.request, dataclasses, gc
from concurrent.futures import ThreadPoolExecutor
import numpy as np, pandas as pd, torch, torchvision
import torch.nn.functional as F
import torch.nn as nn
import inspect
from collections import deque
from pycocotools import mask as mask_utils
import matplotlib.pyplot as plt
import matplotlib as mpl
import pickle
from PIL import Image
import transformers
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

OUT = "/kaggle/working/qual"
os.makedirs(OUT, exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False

ENV = {
    "run_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
    "python": platform.python_version(), "torch": torch.__version__, "torchvision": torchvision.__version__,
    "transformers": transformers.__version__,
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
}
print(json.dumps(ENV, indent=2))
if ENV["gpu"] is None:
    print("\n⚠⚠ NO GPU. Stop, set Accelerator to GPU in the right panel, and Run All again. ⚠⚠")

def sync():
    if DEVICE == "cuda":
        torch.cuda.synchronize()

## 1. Configuration

In [ ]:
CFG = {
    "n_images": 300,
    "seed": 3,
    "batch_size": 8,
    "models": ["detr_r50", "frcnn_v2", "yolos_small", "grounding_dino_tiny"],
    "micro_batch": {"grounding_dino_tiny": 2, "yolos_small": 4},
    "flip_thr_per_model": {"grounding_dino_tiny": 0.3},
    "flip_score_thr": 0.5,
    "flip_iou": 0.9,
    "coco_root": "/kaggle/input",
    "coco_download_dir": "/kaggle/working/coco",
}
AMP = False
# Picked from the first run's contact sheets: clear objects, no close-up faces, both "lost" and "new" boxes where possible.
# Set TEASER_ID = None and GRID_IDS = {} to go back to the automatic ranking.
TEASER_ID = 507042    # giraffes: one giraffe lost, one new giraffe appears; our fix restores batch 1 exactly
PEOPLE_FREE_MATES = True   # False = use the widest/tallest images of the pool even if they show people (the first run's batch-mates)
GRID_IDS = {"detr_r50": [562443, 297396], "frcnn_v2": [367195, 565391],
            "yolos_small": [541634, 313783], "grounding_dino_tiny": [226111, 42178]}
random.seed(CFG["seed"]); np.random.seed(CFG["seed"]); torch.manual_seed(CFG["seed"])

## 2. COCO val2017 (300-image pool)

In [ ]:
def find_attached_coco(root):
    anns = glob.glob(os.path.join(root, "**", "instances_val2017.json"), recursive=True)
    probe = glob.glob(os.path.join(root, "**", "000000000139.jpg"), recursive=True)
    if anns and probe:
        return anns[0], os.path.dirname(probe[0])
    return None, None

ANN, IMG_DIR = find_attached_coco(CFG["coco_root"])
if ANN is None:
    os.makedirs(CFG["coco_download_dir"], exist_ok=True)
    ANN = os.path.join(CFG["coco_download_dir"], "instances_val2017.json")
    IMG_DIR = os.path.join(CFG["coco_download_dir"], "val2017")
    if not os.path.exists(ANN):
        zp = os.path.join(CFG["coco_download_dir"], "ann.zip")
        print("downloading COCO annotations (~250 MB) ...")
        urllib.request.urlretrieve("http://images.cocodataset.org/annotations/annotations_trainval2017.zip", zp)
        with zipfile.ZipFile(zp) as z, open(ANN, "wb") as f:
            f.write(z.read("annotations/instances_val2017.json"))
        os.remove(zp)

with contextlib.redirect_stdout(io.StringIO()):
    GT = COCO(ANN)
NAME2CAT = {c["name"].strip().lower(): c["id"] for c in GT.loadCats(GT.getCatIds())}
IDS = sorted(random.Random(CFG["seed"]).sample(sorted(GT.getImgIds()), CFG["n_images"]))
INFO = GT.loadImgs(IDS)
os.makedirs(IMG_DIR, exist_ok=True)
def fetch(info):
    p = os.path.join(IMG_DIR, info["file_name"])
    if not os.path.exists(p):
        urllib.request.urlretrieve(f"http://images.cocodataset.org/val2017/{info['file_name']}", p)
    return p
with ThreadPoolExecutor(16) as ex:
    paths = list(ex.map(fetch, INFO))
IMGS = [Image.open(p).convert("RGB") for p in paths]
N = len(IMGS)
print(f"{N} images; first ids {IDS[:3]} (Day 0 used seed 0 too, so these are the same images)")

## 3. The `self` fix

In [ ]:
class Fix:
    COMPONENTS = ("conv", "norm", "own", "exactmask")
    def __init__(self):
        for c in self.COMPONENTS:
            setattr(self, c, False)
        self.valid, self.canvas, self.trivial, self._cache = None, None, True, {}

    def set(self, components):
        for c in self.COMPONENTS:
            setattr(self, c, c in components)

    def any_on(self):
        return any(getattr(self, c) for c in self.COMPONENTS)

    def begin(self, valid, canvas):
        self.valid = [tuple(int(v) for v in s) for s in valid]
        self.canvas = tuple(int(c) for c in canvas)
        self.trivial = all(v == self.canvas for v in self.valid)

    def end(self):
        self.valid = None

    def active(self):
        return self.valid is not None and not self.trivial

    def applies(self, x):
        return self.active() and x.dim() == 4 and x.shape[0] == len(self.valid)

    @staticmethod
    def _stride(n, N):
        for s in (1, 2, 4, 8, 16, 32, 64, 128, 256):
            if math.ceil(N / s) == n:
                return s
        return N / n

    def extents(self, H, W):
        """each image's valid (rows, cols) at a feature map of size H x W"""
        sh, sw = self._stride(H, self.canvas[0]), self._stride(W, self.canvas[1])
        return [(math.ceil(h / sh), math.ceil(w / sw)) for h, w in self.valid]

    def mask(self, x):
        H, W = x.shape[-2:]
        key = (H, W, tuple(self.valid), self.canvas, x.device, x.dtype)
        if key not in self._cache:
            m = torch.zeros(len(self.valid), 1, H, W, device=x.device, dtype=x.dtype)
            for i, (h, w) in enumerate(self.extents(H, W)):
                m[i, :, :h, :w] = 1
            if len(self._cache) > 64:
                self._cache.clear()
            self._cache[key] = m
        return self._cache[key]

def _mask_conv_inputs(root, fix):
    """Zero the padding at the input of every conv / max-pool: each layer then sees exactly the zeros its
    batch-1 run sees past the canvas edge (activations are >= 0 after ReLU, so max-pool is unaffected)."""
    def pre(mod, inp):
        x = inp[0]
        if fix.conv and fix.applies(x):
            return (x * fix.mask(x),) + tuple(inp[1:])
    for m in root.modules():
        if isinstance(m, (nn.Conv2d, nn.MaxPool2d)):
            m.register_forward_pre_hook(pre)

def _masked_groupnorm(root, fix):
    """GroupNorm statistics over each image's own valid area only."""
    n = 0
    for mod in root.modules():
        if isinstance(mod, nn.GroupNorm):
            orig = mod.forward
            def fwd(x, mod=mod, orig=orig):
                if not (fix.norm and fix.applies(x)):
                    return orig(x)
                m = fix.mask(x).float(); B, C, H, W = x.shape; G = mod.num_groups
                xg, mg = x.float().reshape(B, G, C // G, H, W), m.reshape(B, 1, 1, H, W)
                cnt = mg.sum(dim=(2, 3, 4), keepdim=True) * (C // G)
                mean = (xg * mg).sum(dim=(2, 3, 4), keepdim=True) / cnt
                var = (((xg - mean) ** 2) * mg).sum(dim=(2, 3, 4), keepdim=True) / cnt
                y = ((xg - mean) / torch.sqrt(var + mod.eps)).reshape(B, C, H, W)
                if mod.affine:
                    y = y * mod.weight.float()[None, :, None, None] + mod.bias.float()[None, :, None, None]
                return y.to(x.dtype)
            mod.forward = fwd; n += 1
    return n

def _suppress_padding_scores(conv, fix):
    """Classification logits at positions that do not exist in the image's batch-1 run -> -1e4."""
    def hook(mod, inp, out):
        if fix.conv and fix.applies(out):
            return out.masked_fill(fix.mask(out) == 0, -1e4)
    conv.register_forward_hook(hook)

def _own_anchors(ag, fix):
    """Each image gets the anchors its own batch-1 canvas would give: torchvision spaces anchors at
    canvas // grid, so we use the image's own canvas and grid. Anchors at positions that do not exist in
    its batch-1 run become zero-size boxes, which torchvision removes."""
    orig = ag.forward
    def fwd(image_list, feature_maps):
        if not (fix.own and fix.active()):
            return orig(image_list, feature_maps)
        grid_sizes = [list(f.shape[-2:]) for f in feature_maps]
        Hc, Wc = image_list.tensors.shape[-2:]
        dtype, device = feature_maps[0].dtype, feature_maps[0].device
        ag.set_cell_anchors(dtype, device)
        n_cell = [ca.shape[0] for ca in ag.cell_anchors]
        out = []
        for ch, cw in fix.valid:
            strides, keep = [], []
            for (gh, gw), a in zip(grid_sizes, n_cell):
                sh, sw = 2 ** round(math.log2(Hc / gh)), 2 ** round(math.log2(Wc / gw))
                oh, ow = math.ceil(ch / sh), math.ceil(cw / sw)
                strides.append([torch.tensor(ch // oh, dtype=torch.int64, device=device),
                                torch.tensor(cw // ow, dtype=torch.int64, device=device)])
                k = torch.zeros(gh, gw, dtype=dtype, device=device); k[:oh, :ow] = 1
                keep.append(k.reshape(-1).repeat_interleave(a))
            anc = ag.grid_anchors(grid_sizes, strides)
            out.append(torch.cat([x * k[:, None] for x, k in zip(anc, keep)]))
        return out
    ag.forward = fwd

def install_fix_tv(model, fix):
    heads = [model.rpn.head] if hasattr(model, "rpn") else [model.head]
    for root in [model.backbone] + heads:
        _mask_conv_inputs(root, fix)
    has_roi = hasattr(model, "roi_heads")
    def zero_outputs(mod, inp, out):
        if not (fix.conv and fix.active()):
            return
        new = type(out)()
        for k, v in out.items():
            v = v * fix.mask(v)
            if fix.own and has_roi:
                # RoIAlign at batch 1 clamps samples at the canvas edge to the last row/column; copying that
                # row/column one step outward reproduces this inside a bigger canvas.
                H, W = v.shape[-2:]
                for i, (hl, wl) in enumerate(fix.extents(H, W)):
                    if wl < W:
                        v[i, :, :hl, wl] = v[i, :, :hl, wl - 1]
                    if hl < H:
                        v[i, :, hl, :min(wl + 1, W)] = v[i, :, hl - 1, :min(wl + 1, W)]
            new[k] = v
        return new
    model.backbone.register_forward_hook(zero_outputs)
    g = sum(_masked_groupnorm(h, fix) for h in heads) + _masked_groupnorm(model.backbone, fix)
    if hasattr(model, "rpn"):
        _own_anchors(model.rpn.anchor_generator, fix); _suppress_padding_scores(model.rpn.head.cls_logits, fix)
    else:
        _own_anchors(model.anchor_generator, fix); _suppress_padding_scores(model.head.classification_head.cls_logits, fix)
    fix.SELF = ("conv", "norm", "own") if g else ("conv", "own")

def install_fix_hf(model, fix):
    enc = model.model.backbone
    _mask_conv_inputs(enc, fix)
    orig = enc.forward
    def fwd(pixel_values, pixel_mask):
        out = orig(pixel_values, pixel_mask)
        if not (fix.exactmask and fix.active()):
            return out
        # the transformer's padding mask with exact extents (the library shrinks it with nearest-neighbour rounding)
        return [(fm, fix.mask(fm)[:, 0].bool()) for fm, _ in out]
    enc.forward = fwd
    fix.SELF = ("conv", "exactmask")

## 4. Detectors

In [ ]:
ALIASES = {"motorbike": "motorcycle", "aeroplane": "airplane", "sofa": "couch", "tvmonitor": "tv", "tv monitor": "tv",
           "diningtable": "dining table", "pottedplant": "potted plant", "hair dryer": "hair drier",
           "sports ball": "sports ball", "cellphone": "cell phone", "mobile phone": "cell phone"}
def _norm(s):
    s = " ".join(str(s).lower().replace("_", " ").replace("-", " ").split())
    return ALIASES.get(s, s)
def names_to_cat(names, label=""):
    m, miss = {}, []
    for k, v in names.items():
        c = NAME2CAT.get(_norm(v))
        if c is not None:
            m[int(k)] = c
        elif _norm(v) not in ("n/a", "__background__", "background", "no object"):
            miss.append(v)
    if miss:
        print(f"  unmapped class names in {label}: {miss}")
    return m

def load_hf_model(cls, repo, revision=None):
    try:
        return cls.from_pretrained(repo, revision=revision)
    except Exception as ex:
        if "alidation" not in f"{type(ex).__name__} {ex}":
            raise
        from huggingface_hub import hf_hub_download
        raw = json.load(open(hf_hub_download(repo, "config.json", revision=revision)))
        defaults = {f.name: f.default for f in dataclasses.fields(cls.config_class) if f.default is not dataclasses.MISSING}
        dropped = [k for k, v in raw.items() if v is None and defaults.get(k) is not None]
        print(f"  (config fix for {repo}: replaced null fields {dropped} with library defaults)")
        raw = {k: v for k, v in raw.items() if k not in dropped}
        return cls.from_pretrained(repo, revision=revision, config=cls.config_class.from_dict(raw))

def roundup(x, d):
    return int(math.ceil(x / d) * d)

def read_csv(path):
    try:
        return pd.read_csv(path)
    except (FileNotFoundError, pd.errors.EmptyDataError):
        return pd.DataFrame()

def _pack(boxes, scores, labels, lab2cat):
    lab = labels.cpu().numpy()
    keep = np.array([l in lab2cat for l in lab], bool)
    return {"boxes": boxes.float().cpu().numpy()[keep], "scores": scores.float().cpu().numpy()[keep],
            "labels": np.array([lab2cat[l] for l in lab[keep]], dtype=int)}

class MicroBatch:
    """If a batch does not fit in GPU memory, split it into smaller pieces that are all padded to the canvas the
    full batch would have had. Images in these models never interact except through padding, so this gives the
    same result as the full batch (verified for DETR by the canvas self-test, and for every Part-B model by B0)."""
    micro = None   # largest piece size known to fit; None = no limit found yet

    def batch_canvas(self, imgs):
        sizes = [self.input_size(im) for im in imgs]
        return (roundup(max(h for h, _ in sizes), self.div), roundup(max(w for _, w in sizes), self.div))

    def run(self, imgs, canvas=None):
        if self.fixed_size:
            return self._run(imgs, canvas)
        if self.micro is not None and len(imgs) > self.micro:
            return self._split(imgs, canvas)
        try:
            return self._run(imgs, canvas)
        except torch.cuda.OutOfMemoryError:
            if len(imgs) == 1:
                raise
        # We are now OUTSIDE the except block, so the failed attempt (and the GPU memory it holds) is released
        # before retrying. Retrying inside the except block keeps that memory alive.
        free_mem()
        self.micro = max(1, len(imgs) // 2)
        print(f"  [{self.name}] batch of {len(imgs)} does not fit in GPU memory → running pieces of {self.micro} "
              f"on the full batch's canvas (equivalent result)")
        return self._split(imgs, canvas)

    def _split(self, imgs, canvas):
        canvas = canvas if canvas is not None else self.batch_canvas(imgs)
        step = self.micro                      # fixed for this call, even if a piece lowers self.micro further
        R, P = [], []
        for k in range(0, len(imgs), step):
            r, p = self.run(imgs[k:k + step], canvas)
            R += r; P += p
        assert len(R) == len(imgs)
        return R, P

class HFDet(MicroBatch):
    kind, div = "hf", 1
    def __init__(self, name, repo, cls_name, revision=None, fixed_size=False, fixable=True, proc_cls="AutoImageProcessor"):
        self.name, self.fixed_size = name, fixed_size
        self.proc = getattr(transformers, proc_cls).from_pretrained(repo, revision=revision)
        self.model = load_hf_model(getattr(transformers, cls_name), repo, revision).to(DEVICE).eval()
        self.lab2cat = names_to_cat(self.model.config.id2label, name)
        self.accepts_mask = "pixel_mask" in inspect.signature(self.model.forward).parameters
        self.fix = Fix(); self.fix.SELF = None
        if fixable and not fixed_size:
            install_fix_hf(self.model, self.fix)

    # --- hooks that model-specific subclasses override ---
    def _improc(self):
        return self.proc
    def _extra(self, n):
        return {}
    def _post(self, out, imgs, kw):
        sizes = torch.tensor([[im.size[1], im.size[0]] for im in imgs])
        res = self.proc.post_process_object_detection(out, threshold=0.0, target_sizes=sizes)
        return [_pack(r["boxes"], r["scores"], r["labels"], self.lab2cat) for r in res]

    def _single(self, img):
        enc = self._improc()(images=[img], return_tensors="pt")
        pv = enc["pixel_values"][0]
        m = enc["pixel_mask"][0] if "pixel_mask" in enc else torch.ones(pv.shape[-2:], dtype=torch.long)
        return pv, m

    def input_size(self, img):
        return tuple(self._single(img)[0].shape[-2:])

    @torch.inference_mode()
    def _run(self, imgs, canvas=None):
        if canvas is None:
            enc = self._improc()(images=imgs, return_tensors="pt")
            pv = enc["pixel_values"]
            mask = enc["pixel_mask"] if "pixel_mask" in enc else None
        else:
            H, W = canvas; pvs, ms = [], []
            for im in imgs:
                p, m = self._single(im); h, w = p.shape[-2:]
                assert h <= H and w <= W, f"canvas {canvas} smaller than image {(h, w)}"
                pvs.append(F.pad(p, (0, W - w, 0, H - h))); ms.append(F.pad(m, (0, W - w, 0, H - h)))
            pv, mask = torch.stack(pvs), torch.stack(ms)
        self._last_mask = mask
        kw = {"pixel_values": pv.to(DEVICE)}
        if mask is not None and self.accepts_mask:
            kw["pixel_mask"] = mask.to(DEVICE)
            if self.fix.any_on():
                self.fix.begin([(int(mask[i, :, 0].sum()), int(mask[i, 0, :].sum())) for i in range(len(imgs))], pv.shape[-2:])
        kw.update(self._extra(len(imgs)))
        try:
            out = self.model(**kw)
        finally:
            self.fix.end()
        pad = (1 - mask.float().mean(dim=(1, 2))).tolist() if mask is not None else [0.0] * len(imgs)
        return self._post(out, imgs, kw), pad


class YolosDet(HFDet):
    """YOLOS has no padding mask. `rescale=True` tests the coordinate-frame hypothesis: if YOLOS predicts boxes
    relative to the padded canvas instead of the image, multiplying them by canvas/image size puts them back."""
    rescale = False
    def __init__(self, name, repo):
        super().__init__(name, repo, "YolosForObjectDetection", fixable=False)
    def _post(self, out, imgs, kw):
        if self.rescale and self._last_mask is not None:
            m = self._last_mask
            H, W = m.shape[-2:]
            h = m[:, :, 0].sum(1).float(); w = m[:, 0, :].sum(1).float()
            s = torch.stack([W / w, H / h, W / w, H / h], dim=1).to(out.pred_boxes.device)
            out.pred_boxes = out.pred_boxes * s[:, None, :]
        return super()._post(out, imgs, kw)

COCO_PROMPT = " . ".join(c["name"] for c in GT.loadCats(GT.getCatIds())) + " ."

class GroundingDINODet(HFDet):
    """Open-vocabulary detector prompted with all 80 COCO class names."""
    def __init__(self, name, repo):
        super().__init__(name, repo, "GroundingDinoForObjectDetection", fixable=False, proc_cls="AutoProcessor")
        self.lab2cat = {}
        n_tok = len(self.proc.tokenizer(COCO_PROMPT)["input_ids"])
        print(f"  prompt: {n_tok} tokens (model limit {getattr(self.model.config, 'max_text_len', '?')})")
    def _improc(self):
        return self.proc.image_processor
    def _extra(self, n):
        t = self.proc.tokenizer([COCO_PROMPT] * n, padding="longest", return_tensors="pt")
        return {k: v.to(DEVICE) for k, v in t.items()}
    def _post(self, out, imgs, kw):
        sizes = [(im.size[1], im.size[0]) for im in imgs]
        res = self.proc.post_process_grounded_object_detection(out, input_ids=kw["input_ids"], threshold=0.05,
                                                              text_threshold=0.25, target_sizes=sizes)
        packed = []
        for r in res:
            names = r.get("text_labels", r.get("labels"))
            cats = [NAME2CAT.get(_norm(n)) for n in names]
            keep = np.array([c is not None for c in cats], bool)
            packed.append({"boxes": r["boxes"].float().cpu().numpy()[keep], "scores": r["scores"].float().cpu().numpy()[keep],
                           "labels": np.array([c for c in cats if c is not None], dtype=int)})
        return packed

class Mask2FormerDet(HFDet):
    """Instance segmentation; boxes are taken from the predicted masks."""
    def __init__(self, name, repo):
        super().__init__(name, repo, "Mask2FormerForUniversalSegmentation", fixable=False)
    def _post(self, out, imgs, kw):
        sizes = [(im.size[1], im.size[0]) for im in imgs]
        res = self.proc.post_process_instance_segmentation(out, threshold=0.1, target_sizes=sizes, return_binary_maps=True)
        packed = []
        for r in res:
            seg = r["segmentation"]; info = r["segments_info"]
            boxes, scores, labels = [], [], []
            if seg is not None and len(info):
                segs = seg.bool().cpu().numpy() if seg.dim() == 3 else None
                for k, s in enumerate(info):
                    m = segs[k] if segs is not None else (seg == s["id"]).cpu().numpy()
                    c = self.lab2cat.get(int(s["label_id"]))
                    if c is None or not m.any():
                        continue
                    ys, xs = np.where(m)
                    boxes.append([xs.min(), ys.min(), xs.max() + 1, ys.max() + 1]); scores.append(s["score"]); labels.append(c)
            packed.append({"boxes": np.array(boxes, np.float32).reshape(-1, 4), "scores": np.array(scores, np.float32),
                           "labels": np.array(labels, int)})
        return packed

class TVDet(MicroBatch):
    kind, div, fixed_size = "tv", 32, False
    with_masks = False
    def __init__(self, name, builder, weights):
        self.name = name
        self.model = builder(weights=weights).to(DEVICE).eval()
        self.tf = weights.transforms()
        self.lab2cat = names_to_cat(dict(enumerate(weights.meta["categories"])), name)
        self._setup()

    def _setup(self):
        self._canvas = None
        self.fix = Fix()
        orig = self.model.transform.batch_images
        def batch_images(images, size_divisible=32):
            b = orig(images, size_divisible)
            H, W = self._canvas if self._canvas is not None else tuple(b.shape[-2:])
            assert b.shape[-2] <= H and b.shape[-1] <= W, f"canvas {(H, W)} smaller than batch {tuple(b.shape[-2:])}"
            if (H, W) == tuple(b.shape[-2:]):
                return b
            return F.pad(b, (0, W - b.shape[-1], 0, H - b.shape[-2]))
        self.model.transform.batch_images = batch_images
        install_fix_tv(self.model, self.fix)

    def _pack_masks(self, o):
        lab = o["labels"].cpu().numpy(); sc = o["scores"].float().cpu().numpy()
        keep = np.array([l in self.lab2cat for l in lab], bool) & (sc >= 0.05)
        ms = (o["masks"][:, 0] > 0.5).cpu().numpy()[keep]
        rles = []
        for m in ms:
            r = mask_utils.encode(np.asfortranarray(m.astype(np.uint8))); r["counts"] = r["counts"].decode("ascii"); rles.append(r)
        return {"boxes": o["boxes"].float().cpu().numpy()[keep], "scores": sc[keep],
                "labels": np.array([self.lab2cat[l] for l in lab[keep]], dtype=int), "masks": rles}

    @torch.inference_mode()
    def input_size(self, img):
        il, _ = self.model.transform([img if torch.is_tensor(img) else self.tf(img).to(DEVICE)])
        return tuple(il.image_sizes[0])

    @torch.inference_mode()
    def _run(self, imgs, canvas=None):
        self._canvas = canvas
        try:
            x = [im if torch.is_tensor(im) else self.tf(im).to(DEVICE) for im in imgs]
            il, _ = self.model.transform(x)
            H, W = il.tensors.shape[-2:]
            pad = [1 - (h * w) / (H * W) for h, w in il.image_sizes]
            if self.fix.any_on():
                d = self.model.transform.size_divisible
                sizes = [tuple(s) for s in il.image_sizes]
                self.fix.begin([(roundup(h, d), roundup(w, d)) for h, w in sizes] if self.fix.own else sizes, (H, W))
            with torch.autocast("cuda", dtype=torch.float16, enabled=AMP and DEVICE == "cuda"):
                out = self.model(x)
        finally:
            self._canvas = None; self.fix.end()
        if self.with_masks:
            return [self._pack_masks(o) for o in out], pad
        return [_pack(o["boxes"], o["scores"], o["labels"], self.lab2cat) for o in out], pad

def _tv(builder_name, weights_name, name):
    import torchvision.models.detection as D
    return lambda: TVDet(name, getattr(D, builder_name), getattr(D, weights_name).DEFAULT)

LOADERS = {
    # models from Exps 1-3 (used here for the mask and speed parts)
    "detr_r50": lambda: HFDet("detr_r50", "facebook/detr-resnet-50", "DetrForObjectDetection", revision="no_timm"),
    "frcnn_v2": _tv("fasterrcnn_resnet50_fpn_v2", "FasterRCNN_ResNet50_FPN_V2_Weights", "frcnn_v2"),
    "maskrcnn_v2": _tv("maskrcnn_resnet50_fpn_v2", "MaskRCNN_ResNet50_FPN_V2_Weights", "maskrcnn_v2"),
    "retinanet_v2": _tv("retinanet_resnet50_fpn_v2", "RetinaNet_ResNet50_FPN_V2_Weights", "retinanet_v2"),
    "deformable_detr": lambda: HFDet("deformable_detr", "SenseTime/deformable-detr", "DeformableDetrForObjectDetection", fixable=False),
    "fcos": _tv("fcos_resnet50_fpn", "FCOS_ResNet50_FPN_Weights", "fcos"),
    "rtdetr_r50": lambda: HFDet("rtdetr_r50", "PekingU/rtdetr_r50vd", "RTDetrForObjectDetection", fixed_size=True),
    # new in Exp 4
    "conditional_detr": lambda: HFDet("conditional_detr", "microsoft/conditional-detr-resnet-50", "ConditionalDetrForObjectDetection"),
    "dab_detr": lambda: HFDet("dab_detr", "IDEA-Research/dab-detr-resnet-50", "DabDetrForObjectDetection", fixable=False),
    "yolos_small": lambda: YolosDet("yolos_small", "hustvl/yolos-small"),
    "grounding_dino_tiny": lambda: GroundingDINODet("grounding_dino_tiny", "IDEA-Research/grounding-dino-tiny"),
    "mask2former_swin_s": lambda: Mask2FormerDet("mask2former_swin_s", "facebook/mask2former-swin-small-coco-instance"),
    "rtdetr_v2_r50": lambda: HFDet("rtdetr_v2_r50", "PekingU/rtdetr_v2_r50vd", "RTDetrV2ForObjectDetection", fixed_size=True),
}
# Published full COCO val2017 box AP; our 500-image batch-1 AP should be within about ±2.
REF_AP = {"deformable_detr": 44.5, "fcos": 39.2, "rtdetr_r50": 53.1, "detr_r50": 42.0, "frcnn_v2": 46.7, "maskrcnn_v2": 47.4, "retinanet_v2": 41.5,
          "conditional_detr": 41.0, "dab_detr": 42.2, "yolos_small": 36.1, "grounding_dino_tiny": 48.4,
          "mask2former_swin_s": None, "rtdetr_v2_r50": 53.4}

def load(name):
    free_mem()
    t0 = time.time()
    try:
        det = LOADERS[name]()
        det.micro = CFG["micro_batch"].get(name)
    except Exception as ex:
        print(f"✗ {name} failed to load: {type(ex).__name__}: {str(ex)[:300]}  — skipping this model")
        return None
    note = f"; runs in pieces of {det.micro} (padded to the full batch's canvas)" if det.micro else ""
    print(f"\n== {name} loaded in {time.time() - t0:.0f}s; mapped classes {len(set(det.lab2cat.values()))}/80{note}")
    return det

def free_mem():
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

## 5. Metrics and helpers

In [ ]:
def iou_matrix(a, b):
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)))
    x1 = np.maximum(a[:, None, 0], b[None, :, 0]); y1 = np.maximum(a[:, None, 1], b[None, :, 1])
    x2 = np.minimum(a[:, None, 2], b[None, :, 2]); y2 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    area = lambda z: (z[:, 2] - z[:, 0]) * (z[:, 3] - z[:, 1])
    return inter / (area(a)[:, None] + area(b)[None, :] - inter + 1e-9)

def greedy_match(p, q, thr, iou_min=0.5):
    ia = np.where(p["scores"] >= thr)[0]; ib = np.where(q["scores"] >= thr)[0]
    ia = ia[np.argsort(-p["scores"][ia])]
    M = iou_matrix(p["boxes"][ia], q["boxes"][ib]); used = set(); pairs = []
    for r, i in enumerate(ia):
        best, bc = -1.0, None
        for c, j in enumerate(ib):
            if c not in used and p["labels"][i] == q["labels"][j] and M[r, c] > best:
                best, bc = M[r, c], c
        if bc is not None and best >= iou_min:
            used.add(bc); pairs.append((i, ib[bc], best))
    return pairs, ia, ib

def compare(ref, new):
    """Image-level flip + detection-level change counts + max score change, ref vs new."""
    pairs, ia, ib = greedy_match(ref, new, CFG["flip_score_thr"])
    stable = sum(iou >= CFG["flip_iou"] for _, _, iou in pairs)
    moved = len(pairs) - stable
    lost, gained = len(ia) - len(pairs), len(ib) - len(pairs)
    allp, _, _ = greedy_match(ref, new, 0.05)
    dscore = max([abs(ref["scores"][i] - new["scores"][j]) for i, j, _ in allp], default=0.0)
    return {"flip": (moved + lost + gained) > 0, "stable": stable, "moved": moved, "lost": lost, "gained": gained,
            "max_score_delta": dscore}

def det_survival(ref, new, W, H):
    """For each confident ref detection: did it survive unchanged? plus its distance to the right/bottom edge."""
    pairs, ia, _ = greedy_match(ref, new, CFG["flip_score_thr"])
    ok = {i for i, _, iou in pairs if iou >= CFG["flip_iou"]}
    rows = []
    for i in ia:
        x1, y1, x2, y2 = ref["boxes"][i]
        rows.append({"changed": i not in ok, "d_right": (W - x2) / W, "d_bottom": (H - y2) / H,
                     "rel_area": (x2 - x1) * (y2 - y1) / (W * H), "score": ref["scores"][i]})
    return rows

def coco_ap(preds, idx=None):
    idx = list(range(N)) if idx is None else idx
    dets = []
    for k in idx:
        r = preds[k]; order = np.argsort(-r["scores"])[:100]
        for t in order:
            x1, y1, x2, y2 = r["boxes"][t]
            dets.append({"image_id": IDS[k], "category_id": int(r["labels"][t]),
                         "bbox": [float(x1), float(y1), float(x2 - x1), float(y2 - y1)], "score": float(r["scores"][t])})
    if not dets:
        return 0.0
    with contextlib.redirect_stdout(io.StringIO()):
        dt = GT.loadRes(dets); E = COCOeval(GT, dt, "bbox"); E.params.imgIds = [IDS[k] for k in idx]
        E.evaluate(); E.accumulate(); E.summarize()
    return 100 * E.stats[0]

def run_batches(det, batches, canvas_fn=None, targets=None):
    """batches: list of index lists. canvas_fn(batch)->(H,W) or None. targets: optional list of index lists
    saying which members of each batch to record (others are companions). Returns preds, pads, seconds."""
    P, PD = [None] * N, [None] * N
    sync(); t0 = time.time()
    for bi, b in enumerate(batches):
        r, p = det.run([IMGS[i] for i in b], canvas=canvas_fn(b) if canvas_fn else None)
        keep = set(targets[bi]) if targets else set(b)
        for i, ri, pi in zip(b, r, p):
            if i in keep and P[i] is None:
                P[i], PD[i] = ri, pi
    sync()
    return P, PD, time.time() - t0

In [ ]:
B = CFG["batch_size"]
def chunks(order, k=B):
    order = list(order)
    return [order[i:i + k] for i in range(0, len(order), k)]

def run_on(det, batches, source, targets=None):
    """like run_batches, but images come from `source` (PIL images or GPU tensors)"""
    P, PD = [None] * N, [None] * N
    sync(); t0 = time.time()
    for bi, b in enumerate(batches):
        r, p = det.run([source[i] for i in b])
        keep = set(targets[bi]) if targets else set(b)
        for i, ri, pi in zip(b, r, p):
            if i in keep and P[i] is None:
                P[i], PD[i] = ri, pi
    sync()
    return P, PD, time.time() - t0

def warmup(det, source=None):
    source = IMGS if source is None else source
    for _ in range(2):
        det.run(source[:B]); det.run(source[:1])
    sync()

def with_thr(model, thr=None):
    CFG["flip_score_thr"] = thr if thr is not None else CFG["flip_thr_per_model"].get(model, 0.5)

def mask_compare(ref, new):
    c = compare(ref, new)
    pairs, _, _ = greedy_match(ref, new, CFG["flip_score_thr"])
    ious = [float(mask_utils.iou([new["masks"][j]], [ref["masks"][i]], [0])[0][0]) for i, j, _ in pairs]
    return {"flip": c["flip"], "mask_flip": bool(c["flip"] or any(v < CFG["flip_iou"] for v in ious)),
            "min_mask_iou": min(ious, default=1.0), "max_score_delta": c["max_score_delta"]}

def coco_segm_ap(preds):
    dets = []
    for k, img_id in enumerate(IDS):
        r = preds[k]
        for t in np.argsort(-r["scores"])[:100]:
            dets.append({"image_id": img_id, "category_id": int(r["labels"][t]), "segmentation": r["masks"][t],
                         "score": float(r["scores"][t])})
    with contextlib.redirect_stdout(io.StringIO()):
        dt = GT.loadRes(dets); E = COCOeval(GT, dt, "segm"); E.params.imgIds = IDS
        E.evaluate(); E.accumulate(); E.summarize()
    return 100 * E.stats[0]

bs1 = [[i] for i in range(N)]
rand0 = chunks(np.random.default_rng(0).permutation(N))
aspect = np.array([im.size[0] / im.size[1] for im in IMGS]); area = np.array([im.size[0] * im.size[1] for im in IMGS])
sorted_b = chunks(np.lexsort((area, aspect)))
WIDEST, TALLEST = int(np.argmax(aspect)), int(np.argmin(aspect))
mp = chunks(range(N), B - 2); maxpad = [c + [WIDEST, TALLEST] for c in mp]

## 6. Run every model: alone, in a padded batch, in the batch + fix

In [ ]:
# batch-mates: the widest and the tallest image of the pool -> the canvas is as large as it gets
aspect = np.array([im.size[0] / im.size[1] for im in IMGS])
PERSON = 1   # COCO category id of "person"
no_people = [i for i in range(N) if not GT.getAnnIds(imgIds=[IDS[i]], catIds=[PERSON])]
if len(no_people) < 3 or not PEOPLE_FREE_MATES:
    no_people = list(range(N))
by_wide = sorted(no_people, key=lambda i: -aspect[i]); by_tall = sorted(no_people, key=lambda i: aspect[i])
def mates(i):
    """the widest and the tallest image of the pool without people (so the figure shows no faces)"""
    w = next(k for k in by_wide if k != i); t = next(k for k in by_tall if k != i)
    return [w, t]

CACHE = f"{OUT}/runs_mates{IDS[by_wide[0]]}_{IDS[by_tall[0]]}.pkl"     # a new choice of batch-mates gets a new cache
RUNS = pickle.load(open(CACHE, "rb")) if os.path.exists(CACHE) else {}
for mname in CFG["models"]:
    if mname in RUNS:
        print(f"== {mname}: cached"); continue
    det = None; free_mem()
    det = load(mname)
    if det is None:
        continue
    warmup(det)
    fixname = "fix" if det.fix.SELF else ("rescale" if hasattr(det, "rescale") else None)
    out = {"alone": [None] * N, "batch": [None] * N, fixname: [None] * N} if fixname else {"alone": [None] * N, "batch": [None] * N}
    canvas = [None] * N
    t0 = time.time()
    for i in range(N):
        b = [i] + mates(i)
        det.fix.set(())
        if hasattr(det, "rescale"):
            det.rescale = False
        out["alone"][i] = det.run([IMGS[i]])[0][0]
        r, _ = det.run([IMGS[k] for k in b]); out["batch"][i] = r[0]
        sizes = [det.input_size(IMGS[k]) for k in b]
        own = det.input_size(IMGS[i])
        canvas[i] = (roundup(max(s[0] for s in sizes), det.div) / own[0],
                     roundup(max(s[1] for s in sizes), det.div) / own[1])    # padded canvas / image, per axis
        if fixname == "fix":
            det.fix.set(det.fix.SELF); r, _ = det.run([IMGS[k] for k in b]); out["fix"][i] = r[0]; det.fix.set(())
        elif fixname == "rescale":
            det.rescale = True; r, _ = det.run([IMGS[k] for k in b]); out["rescale"][i] = r[0]; det.rescale = False
        if i % 50 == 0:
            print(f"  {mname}: {i + 1}/{N}  ({time.time() - t0:.0f}s)")
    RUNS[mname] = {"out": out, "canvas": canvas, "fixname": fixname}
    det = None; free_mem()
    pickle.dump(RUNS, open(CACHE, "wb"))
print("done:", list(RUNS))

## 7. Rank images by how visibly the detections change

Re-run from here after changing `TEASER_ID` / `GRID_IDS`; the model outputs are cached.

In [ ]:
CATNAME = {c["id"]: c["name"] for c in GT.loadCats(GT.getCatIds())}

def diff(ref, new, thr):
    """split confident detections into unchanged / lost / new (moved boxes count as one lost + one new)"""
    pairs, ia, ib = greedy_match(ref, new, thr)
    keep_a = {i for i, j, iou in pairs if iou >= CFG["flip_iou"]}; keep_b = {j for i, j, iou in pairs if iou >= CFG["flip_iou"]}
    return {"same": [i for i in ia if i in keep_a], "lost": [i for i in ia if i not in keep_a],
            "new": [j for j in ib if j not in keep_b], "same_b": [j for j in ib if j in keep_b]}

def rank(mname):
    R = RUNS[mname]; thr = CFG["flip_thr_per_model"].get(mname, 0.5); rows = []
    for i in range(N):
        a, b = R["out"]["alone"][i], R["out"]["batch"][i]
        d = diff(a, b, thr)
        W, H = IMGS[i].size
        area = lambda r, k: (r["boxes"][k, 2] - r["boxes"][k, 0]) * (r["boxes"][k, 3] - r["boxes"][k, 1]) / (W * H)
        changed = [x for x in [area(a, k) for k in d["lost"]] + [area(b, k) for k in d["new"]] if 0.02 <= x <= 0.5]
        n_conf = len(d["same"]) + len(d["lost"])
        fix_ok = True
        if R["fixname"] == "fix":
            fix_ok = not compare(a, R["out"]["fix"][i])["flip"]
        rows.append({"idx": i, "image_id": IDS[i], "n_conf": n_conf, "n_lost": len(d["lost"]), "n_new": len(d["new"]),
                     "max_changed_area": max(changed, default=0.0), "fix_exact": fix_ok,
                     "canvas_pad": 1 - 1 / (R["canvas"][i][0] * R["canvas"][i][1])})
    df = pd.DataFrame(rows)
    good = (df.n_conf.between(2, 10)) & (df.n_lost + df.n_new >= 1) & df.fix_exact
    df["score"] = np.where(good, df.max_changed_area + 0.02 * (df.n_lost + df.n_new), -1)
    return df.sort_values("score", ascending=False)

RANK = {m: rank(m) for m in RUNS}
for m, df in RANK.items():
    df.to_csv(f"{OUT}/candidates_{m}.csv", index=False)
    print(f"{m}: {int((df.score > 0).sum())} usable candidates; top ids {df.image_id.head(6).tolist()}")

## 8. Drawing helpers

In [ ]:
# one neutral colour for unchanged boxes in every model, so "lost" (red, dashed) and "new" (amber) always stand out
COLOR = {m: "#0072B2" for m in ["detr_r50", "frcnn_v2", "yolos_small", "grounding_dino_tiny"]}
LOST, NEW = "#D62728", "#FFB000"
PRETTY = {"detr_r50": "DETR", "frcnn_v2": "Faster R-CNN", "yolos_small": "YOLOS-S", "grounding_dino_tiny": "Grounding DINO"}
mpl.rcParams.update({"font.family": "sans-serif", "font.sans-serif": ["DejaVu Sans"], "font.size": 7, "hatch.linewidth": 0.4,
                     "pdf.fonttype": 42, "savefig.bbox": "tight", "savefig.pad_inches": 0.02})

def _placed(ax):
    if not hasattr(ax, "_label_boxes"):
        ax._label_boxes = []
    return ax._label_boxes

LABEL_FS = 5.4          # box-label font size (pt); the teaser uses a larger value
PAD_FS, BADGE_FS = 6.5, 7.0

def label(ax, x, y, text, color, H, anchor="top"):
    """Label at a box corner. Labels never overlap: a label that would hit an earlier one is moved
    down (top-anchored) or up (bottom-anchored) until it is free."""
    fs = LABEL_FS
    bb = ax.get_window_extent(); (x0, x1), (y0, y1) = ax.get_xlim(), ax.get_ylim()
    sx, sy = abs(x1 - x0) / bb.width, abs(y1 - y0) / bb.height          # data units per pixel
    dpi = ax.figure.dpi
    w = len(text) * fs * 0.62 * dpi / 72 * sx; h = fs * 1.55 * dpi / 72 * sy
    ytop = y if anchor == "top" else y - h                                # top edge of the label (image y grows down)
    if anchor == "top" and y < 0.05 * H:
        ytop = y + 0.01 * H
    step = h if anchor == "top" else -h
    for _ in range(12):
        if not any(not (x + w < a or b < x or ytop + h < c or d < ytop) for a, b, c, d in _placed(ax)):
            break
        ytop += step
    _placed(ax).append((x, x + w, ytop, ytop + h))
    ax.text(x, ytop, text, fontsize=fs, color="white", va="top", ha="left", clip_on=True, zorder=5,
            bbox=dict(boxstyle="square,pad=0.12", fc=color, ec="none", alpha=0.92))

def draw_boxes(ax, r, idxs, color, H, ls="-", lw=1.1, tag=None, text=True, score=True, anchor="top"):
    for k in idxs:
        x1, y1, x2, y2 = r["boxes"][k]
        ax.add_patch(mpl.patches.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, ec=color, lw=lw, ls=ls, zorder=4))
        if text:
            name = CATNAME.get(int(r["labels"][k]), "?")
            label(ax, x1, y1 if anchor == "top" else y2, f"{tag + ': ' if tag else ''}{name}" +
                  (f" {r['scores'][k]:.2f}" if score else ""), color, H, anchor=anchor)

def panel(ax, mname, i, kind, labels="all"):
    """kind: 'alone' | 'batch' | 'fix' | 'rescale'. labels: 'all' | 'changed' (only lost/new, no scores) | 'none'"""
    txt_same = labels == "all"; txt_chg = labels in ("all", "changed", "changed+score")
    sc = labels in ("all", "changed+score")
    R = RUNS[mname]; thr = CFG["flip_thr_per_model"].get(mname, 0.5)
    im = IMGS[i]; W, H = im.size; a = R["out"]["alone"][i]
    ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values():
        s.set_linewidth(0.5); s.set_color("0.6")
    if kind == "alone":
        ax.imshow(im, extent=(0, W, H, 0)); ax.set_xlim(0, W); ax.set_ylim(H, 0)
        draw_boxes(ax, a, list(np.where(a["scores"] >= thr)[0]), COLOR[mname], H, lw=0.9, text=txt_same, score=sc); return
    b = R["out"][kind][i]; d = diff(a, b, thr)
    if kind == "batch":
        cy, cx = R["canvas"][i]; Wc, Hc = W * cx, H * cy
        ax.add_patch(mpl.patches.Rectangle((0, 0), Wc, Hc, fc="#f2f2f2", ec="#b5b5b5", hatch="////", lw=0, zorder=0))
        ax.imshow(im, extent=(0, W, H, 0), zorder=1); ax.set_xlim(0, Wc); ax.set_ylim(Hc, 0)
        if Wc > W * 1.05:
            ax.text((W + Wc) / 2, Hc / 2, "padding", fontsize=PAD_FS, color="0.35", ha="center", va="center", rotation=90)
        elif Hc > H * 1.05:
            ax.text(Wc / 2, (H + Hc) / 2, "padding", fontsize=PAD_FS, color="0.35", ha="center", va="center")
    else:
        ax.imshow(im, extent=(0, W, H, 0)); ax.set_xlim(0, W); ax.set_ylim(H, 0)
    draw_boxes(ax, b, d["same_b"], COLOR[mname], H, lw=0.9, text=txt_same, score=sc)
    draw_boxes(ax, b, d["new"], NEW, H, lw=2.0, tag="new", text=txt_chg, score=sc)
    draw_boxes(ax, a, d["lost"], LOST, H, ls=(0, (3, 1.5)), lw=1.8, tag="lost", text=txt_chg, score=sc, anchor="bottom")
    if kind == "fix" and not d["lost"] and not d["new"]:
        ax.text(0.97, 0.97, "✓ identical to (a)", transform=ax.transAxes, ha="right", va="top", fontsize=BADGE_FS, zorder=6,
                color="white", bbox=dict(boxstyle="round,pad=0.25", fc="#009E73", ec="none"))

def pick(mname, n=1, exclude=()):
    forced = GRID_IDS.get(mname, [])
    ok = set(RANK[mname][RANK[mname].score > 0].image_id)
    for x in forced:
        if x not in ok:
            print(f"  note: {mname} #{x} no longer shows a clear change with these batch-mates; picking automatically instead")
    ids = [IDS.index(x) for x in forced if x in IDS and x in ok]
    for strict in (True, False):          # fall back to the biggest visible change if nothing passes the filters
        for _, r in RANK[mname].iterrows():
            if len(ids) >= n:
                break
            if (r.score > 0 or not strict) and int(r.idx) not in ids and r.image_id not in exclude:
                ids.append(int(r.idx))
    return ids[:n]

## 9. Teaser (Figure 1)

In [ ]:
m = "detr_r50"
ok = set(RANK[m][RANK[m].score > 0].image_id)
if TEASER_ID is not None and TEASER_ID not in ok:
    print(f"note: teaser #{TEASER_ID} no longer shows a clear change with these batch-mates; picking automatically")
ti = IDS.index(TEASER_ID) if TEASER_ID in ok else pick(m, 1)[0]
LABEL_FS, PAD_FS, BADGE_FS = 7.0, 8.0, 8.0              # readable at full CVPR text width
W, H = IMGS[ti].size; cy, cx = RUNS[m]["canvas"][ti]
ratios = [W / H, (W * cx) / (H * cy), W / H, 0.40]
fig = plt.figure(figsize=(6.875, 6.875 / (sum(ratios) + 0.3) * 1.10))
gs = fig.add_gridspec(1, 4, width_ratios=ratios, wspace=0.07)
axes = [fig.add_subplot(gs[0, k]) for k in range(3)]
for ax, kind, title in zip(axes, ["alone", "batch", "fix"],
                           ["(a) Alone", "(b) In a padded batch", "(c) Batch + our fix"]):
    panel(ax, m, ti, kind, labels="changed+score" if kind == "batch" else "none")
    ax.set_title(title, fontsize=9, loc="left", weight="bold", pad=4)
sub = gs[0, 3].subgridspec(2, 1, hspace=0.15)
for k, j in enumerate(mates(ti)):
    ax = fig.add_subplot(sub[k, 0]); ax.imshow(IMGS[j]); ax.set_xticks([]); ax.set_yticks([])
    for s_ in ax.spines.values():
        s_.set_linewidth(0.5); s_.set_color("0.6")
    if k == 0:
        ax.set_title("Batch-mates", fontsize=8.5, loc="left", pad=4)
handles = [mpl.lines.Line2D([], [], color=COLOR[m], lw=1.4, label="unchanged"),
           mpl.lines.Line2D([], [], color=LOST, lw=2.0, ls=(0, (3, 1.5)), label="lost in the batch"),
           mpl.lines.Line2D([], [], color=NEW, lw=2.2, label="new in the batch"),
           mpl.patches.Patch(fc="#eeeeee", ec="0.6", hatch="////", label="padding")]
hgt = fig.get_figheight()
fig.legend(handles=handles, loc="upper center", ncol=4, fontsize=8, frameon=False, bbox_to_anchor=(0.47, 0.1 - 0.02 / hgt),
           handlelength=2.2, columnspacing=1.6)
fig.savefig(f"{OUT}/teaser.pdf"); fig.savefig(f"{OUT}/teaser.png", dpi=300); plt.show()
LABEL_FS, PAD_FS, BADGE_FS = 5.4, 6.5, 7.0
print("teaser image id:", IDS[ti])

## 10. Supplement grid

In [ ]:
models = [m for m in ["detr_r50", "frcnn_v2", "yolos_small", "grounding_dino_tiny"] if m in RANK]
fig = plt.figure(figsize=(6.875, 1.6 * len(models)))
gs = fig.add_gridspec(len(models), 5, width_ratios=[1, 1, 0.1, 1, 1], wspace=0.04, hspace=0.1)
used = {IDS[ti]}
for r, m in enumerate(models):
    ex = pick(m, 2, exclude=used)
    for c, i in enumerate(ex):
        used.add(IDS[i])
        for k, kind in enumerate(["alone", "batch"]):
            ax = fig.add_subplot(gs[r, 3 * c + k])
            panel(ax, m, i, kind, labels="none" if (kind == "alone" or m == "yolos_small") else "changed")
            if r == 0:
                ax.set_title("alone" if kind == "alone" else "in a padded batch", fontsize=7)
            if kind == "alone":
                ax.text(0.02, 0.02, f"#{IDS[i]}", transform=ax.transAxes, fontsize=5, color="white", va="bottom",
                        bbox=dict(boxstyle="square,pad=0.15", fc="black", ec="none", alpha=0.55))
                if c == 0:
                    ax.set_ylabel(PRETTY[m], fontsize=7)
handles = [mpl.lines.Line2D([], [], color=COLOR["detr_r50"], lw=1.2, label="unchanged"),
           mpl.lines.Line2D([], [], color=LOST, lw=1.3, ls=(0, (3, 1.5)), label="lost in the batch"),
           mpl.lines.Line2D([], [], color=NEW, lw=1.5, label="new in the batch"),
           mpl.patches.Patch(fc="#eeeeee", ec="0.6", hatch="////", label="padding")]
fig.legend(handles=handles, loc="upper center", ncol=4, fontsize=6.3, frameon=False, bbox_to_anchor=(0.5, 0.08))
fig.savefig(f"{OUT}/supp_grid.pdf"); fig.savefig(f"{OUT}/supp_grid.png", dpi=250); plt.show()

## 11. Contact sheets for picking by hand

In [ ]:
for m, df in RANK.items():
    top = [int(x) for x in df[df.score > 0].idx.head(20)] or [int(x) for x in df.idx.head(10)]
    if not top:
        continue
    cols = 5; rows = int(np.ceil(len(top) / cols))
    fig, axes = plt.subplots(rows, cols * 2, figsize=(14, 1.6 * rows))
    axes = np.atleast_2d(axes)
    for ax in axes.ravel():
        ax.axis("off")
    for n, i in enumerate(top):
        rr, cc = divmod(n, cols)
        panel(axes[rr, 2 * cc], m, i, "alone"); panel(axes[rr, 2 * cc + 1], m, i, "batch")
        axes[rr, 2 * cc].set_title(f"#{IDS[i]}", fontsize=7, loc="left")
    fig.suptitle(f"{PRETTY[m]}: top candidates (left: alone, right: in a padded batch)", fontsize=9)
    fig.savefig(f"{OUT}/contact_{m}.png", dpi=110); plt.close(fig)
print("saved:", sorted(os.listdir(OUT)))

## What to bring back

Download the `qual` folder (Output tab or the file browser on the right) and upload `teaser.pdf`, `teaser.png`, `supp_grid.pdf` and the `contact_*.png` sheets here. `teaser.pdf` drops straight into the paper as `fig/teaser.tex`'s image.